# AP01 - Introduction to LlamaIndex

### Build your first RAG application in minutes

**Duration:** 2 hours

**What you'll learn:**
- What is LlamaIndex and when to use it
- LlamaIndex vs LangChain comparison
- Core concepts: Documents, Nodes, Indexes
- Loading and indexing documents
- Basic querying and retrieval
- Building a simple Q&A application

**Prerequisites:**
- Completed Intermediate (I16-I18) - Vector databases and basic RAG
- Understanding of embeddings and similarity search
- Basic Python knowledge

---

## 🔧 Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -qU llama-index llama-index-llms-openai llama-index-embeddings-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# For Google Colab
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Verify API key
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY environment variable"
print("✅ Environment setup complete")

---

## 1. What is LlamaIndex?

### Overview

**LlamaIndex** (formerly GPT Index) is a data framework designed specifically for building Retrieval-Augmented Generation (RAG) applications with Large Language Models.

### Key Philosophy

- **Data-First:** Optimized for connecting LLMs to your data
- **Simple API:** Get started in 5 lines of code
- **Flexible:** Advanced customization when needed
- **Production-Ready:** Built for real applications

### When to Use LlamaIndex

✅ **Use LlamaIndex when:**
- Building RAG applications
- Need to query structured/unstructured data
- Want simple, intuitive API
- Focus is on data ingestion and retrieval

❌ **Consider LangChain when:**
- Building complex multi-agent systems
- Need extensive tool integrations
- Require complex workflows and chains
- Focus is on agent orchestration

### LlamaIndex vs LangChain

| Feature | LlamaIndex | LangChain |
|---------|------------|----------|
| **Primary Focus** | Data ingestion & retrieval | Agent orchestration |
| **Best For** | RAG applications | Complex workflows |
| **Learning Curve** | Easier | Steeper |
| **Setup Time** | Minutes | Hours |
| **Flexibility** | Data queries | General purpose |
| **Documentation** | Excellent, focused | Extensive, broad |

> 💡 **Pro Tip:** You can use both! LlamaIndex for data, LangChain for agents. See AP28 for integration patterns.

---

## 2. Core Concepts

### The LlamaIndex Architecture

```
Documents → Nodes → Index → Query Engine → Response
```

### 1. Documents
- Raw data containers (text, PDF, SQL, etc.)
- Include metadata for filtering

### 2. Nodes
- Chunks of documents
- Atomic units of data
- Connected in a graph structure

### 3. Indexes
- Data structures for efficient retrieval
- VectorStoreIndex, ListIndex, TreeIndex, etc.

### 4. Query Engine
- Interface for asking questions
- Handles retrieval and synthesis

### 5. Response
- Generated answer
- Includes source nodes for citations

---

## 3. Your First LlamaIndex Application

### Simple 5-Line RAG System

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Create sample documents
!mkdir -p data
with open("data/document1.txt", "w") as f:
    f.write("""
    LlamaIndex is a data framework for building LLM applications.
    It provides tools for data ingestion, indexing, and querying.
    LlamaIndex was formerly known as GPT Index.
    """)

with open("data/document2.txt", "w") as f:
    f.write("""
    Retrieval-Augmented Generation (RAG) combines retrieval with generation.
    It allows LLMs to access external knowledge bases.
    RAG improves factuality and reduces hallucinations.
    """)

print("✅ Sample documents created")

In [ ]:
# The famous 5 lines!
documents = SimpleDirectoryReader("data").load_data()  # 1. Load
index = VectorStoreIndex.from_documents(documents)     # 2. Index
query_engine = index.as_query_engine()                 # 3. Create engine
response = query_engine.query("What is LlamaIndex?")  # 4. Query
print(response)                                         # 5. Get answer

print("\n✅ Your first RAG query complete!")

### What Just Happened?

1. **Loaded documents** from directory
2. **Created embeddings** for each chunk
3. **Built vector index** for similarity search
4. **Retrieved relevant chunks** for your query
5. **Generated answer** using LLM + context

---

## 4. Documents and Loaders

### Creating Documents Manually

In [ ]:
from llama_index.core import Document

# Create documents from text
doc1 = Document(
    text="Python is a high-level programming language.",
    metadata={"source": "programming", "language": "python"}
)

doc2 = Document(
    text="JavaScript is used for web development.",
    metadata={"source": "programming", "language": "javascript"}
)

print(f"Doc 1: {doc1.text[:50]}...")
print(f"Metadata: {doc1.metadata}")
print(f"\nDoc 2: {doc2.text[:50]}...")
print(f"Metadata: {doc2.metadata}")

### Loading from Different Sources

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load all files from directory
documents = SimpleDirectoryReader("data").load_data()

print(f"Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Text length: {len(doc.text)} characters")
    print(f"  Preview: {doc.text[:100]}...")

### Common Document Loaders

```python
# PDF files
from llama_index.readers.file import PDFReader
documents = PDFReader().load_data("document.pdf")

# Web pages
from llama_index.readers.web import SimpleWebPageReader
documents = SimpleWebPageReader().load_data(["https://example.com"])

# Databases
from llama_index.readers.database import DatabaseReader
reader = DatabaseReader(uri="postgresql://...")
documents = reader.load_data(query="SELECT * FROM table")

# Notion
from llama_index.readers.notion import NotionPageReader
reader = NotionPageReader(integration_token="...")
documents = reader.load_data(page_ids=["..."])
```

---

## 5. Indexes - The Heart of LlamaIndex

### VectorStoreIndex (Most Common)

In [ ]:
from llama_index.core import VectorStoreIndex, Document

# Create sample documents about AI
docs = [
    Document(text="Machine learning is a subset of artificial intelligence."),
    Document(text="Deep learning uses neural networks with multiple layers."),
    Document(text="Natural language processing helps computers understand text."),
    Document(text="Computer vision enables machines to interpret images."),
]

# Build vector index
index = VectorStoreIndex.from_documents(docs)

print("✅ Vector index created")
print(f"   Indexed {len(docs)} documents")

### Querying the Index

In [ ]:
# Create query engine
query_engine = index.as_query_engine(
    similarity_top_k=2  # Return top 2 most relevant chunks
)

# Ask questions
questions = [
    "What is deep learning?",
    "How do computers understand text?",
    "What is machine learning?"
]

for question in questions:
    print(f"\n❓ Question: {question}")
    response = query_engine.query(question)
    print(f"💡 Answer: {response}")

### Accessing Source Nodes

In [ ]:
# Query and examine sources
response = query_engine.query("What is NLP?")

print("📝 Answer:", response)
print("\n📚 Source nodes:")

for i, node in enumerate(response.source_nodes):
    print(f"\nSource {i+1}:")
    print(f"  Text: {node.text}")
    print(f"  Score: {node.score:.4f}")

---

## 6. Customizing Settings

### Global Settings

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM
Settings.llm = OpenAI(
    model="gpt-4",
    temperature=0.1,
    max_tokens=512
)

# Configure embeddings
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# Configure chunk size
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("✅ Custom settings configured")
print(f"   LLM: {Settings.llm.model}")
print(f"   Embed model: {Settings.embed_model.model_name}")
print(f"   Chunk size: {Settings.chunk_size}")

### Per-Query Settings

In [ ]:
# Create index with default settings
index = VectorStoreIndex.from_documents(docs)

# Query with custom settings
query_engine = index.as_query_engine(
    similarity_top_k=3,           # Number of chunks to retrieve
    response_mode="compact",       # How to synthesize response
    streaming=False                # Enable/disable streaming
)

response = query_engine.query("Explain AI concepts")
print(response)

---

## 7. Response Modes

### Available Response Modes

1. **`compact`** (default) - Concatenate chunks into one prompt
2. **`refine`** - Iteratively refine answer across chunks
3. **`tree_summarize`** - Build tree summary from chunks
4. **`simple_summarize`** - Concatenate and summarize
5. **`no_text`** - Only return retrieved chunks (no LLM)

In [ ]:
# Compare response modes
modes = ["compact", "refine", "tree_summarize"]

question = "What are the main AI concepts discussed?"

for mode in modes:
    query_engine = index.as_query_engine(
        response_mode=mode,
        similarity_top_k=3
    )
    
    response = query_engine.query(question)
    
    print(f"\n{'='*60}")
    print(f"Mode: {mode.upper()}")
    print(f"{'='*60}")
    print(response)

---

## 8. Streaming Responses

In [ ]:
# Enable streaming
query_engine = index.as_query_engine(streaming=True)

# Stream response
print("Streaming response: ", end="")
streaming_response = query_engine.query(
    "Explain machine learning in simple terms"
)

for text in streaming_response.response_gen:
    print(text, end="", flush=True)
print("\n\n✅ Streaming complete")

---

## 9. Practical Example: Document Q&A System

### Create Knowledge Base

In [ ]:
# Create a small knowledge base about Python
python_docs = [
    Document(
        text="""Python is a high-level, interpreted programming language known for its
        simplicity and readability. It was created by Guido van Rossum and first
        released in 1991. Python emphasizes code readability with significant whitespace.""",
        metadata={"category": "introduction", "difficulty": "beginner"}
    ),
    Document(
        text="""Python supports multiple programming paradigms including procedural,
        object-oriented, and functional programming. It has a comprehensive standard
        library and a large ecosystem of third-party packages available through PyPI.""",
        metadata={"category": "features", "difficulty": "intermediate"}
    ),
    Document(
        text="""Common Python applications include web development with Django and Flask,
        data science with NumPy and Pandas, machine learning with TensorFlow and PyTorch,
        and automation scripting. Python is also popular for scientific computing.""",
        metadata={"category": "applications", "difficulty": "intermediate"}
    ),
    Document(
        text="""Python 3 introduced many improvements over Python 2, including better Unicode
        support, type hints, and async/await syntax. Python 2 reached end-of-life in 2020.
        Modern Python development should use Python 3.8 or later.""",
        metadata={"category": "versions", "difficulty": "intermediate"}
    )
]

print(f"✅ Created knowledge base with {len(python_docs)} documents")

### Build Index and Query

In [ ]:
# Build index
python_index = VectorStoreIndex.from_documents(python_docs)

# Create query engine
python_qa = python_index.as_query_engine(
    similarity_top_k=2,
    response_mode="compact"
)

# Interactive Q&A
questions = [
    "Who created Python and when?",
    "What is Python used for?",
    "What are the differences between Python 2 and 3?",
    "What programming paradigms does Python support?"
]

for question in questions:
    print(f"\n{'='*70}")
    print(f"❓ Q: {question}")
    print(f"{'='*70}")
    
    response = python_qa.query(question)
    print(f"💡 A: {response}")
    
    print(f"\n📚 Sources used:")
    for i, node in enumerate(response.source_nodes):
        metadata = node.metadata
        print(f"   {i+1}. Category: {metadata['category']}, "
              f"Difficulty: {metadata['difficulty']}, "
              f"Relevance: {node.score:.3f}")

---

## 10. Saving and Loading Indexes

### Persist to Disk

In [ ]:
# Save index
python_index.storage_context.persist(persist_dir="./storage/python_index")

print("✅ Index saved to ./storage/python_index")

### Load from Disk

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# Load index
storage_context = StorageContext.from_defaults(
    persist_dir="./storage/python_index"
)
loaded_index = load_index_from_storage(storage_context)

# Verify it works
query_engine = loaded_index.as_query_engine()
response = query_engine.query("What is Python?")

print("✅ Index loaded successfully")
print(f"\n💡 Response: {response}")

---

## 🎯 Practice Exercises

### Exercise 1: Personal Knowledge Base
Create a Q&A system for your favorite topic (sports, movies, history, etc.)
- Create 5+ documents
- Add meaningful metadata
- Test with various questions

### Exercise 2: Compare Response Modes
- Use the same query with all response modes
- Compare outputs and performance
- Document which mode works best for what

### Exercise 3: Metadata Filtering
- Add category metadata to documents
- Implement filtered queries (hint: see AP02)
- Test category-specific questions

### Exercise 4: Build a Mini Documentation Bot
- Load actual documentation files
- Create an interactive Q&A system
- Add streaming for better UX

### Challenge: Multi-Source RAG
- Combine documents from different sources
- Add source attribution in responses
- Implement confidence scoring

---

## 📚 Key Takeaways

1. **LlamaIndex is RAG-focused** - Simpler and faster for data-centric apps
2. **5 lines to RAG** - Load, index, query in minutes
3. **Documents → Nodes → Index** - Core data flow
4. **Multiple response modes** - Choose based on your needs
5. **Metadata is powerful** - Use it for filtering and attribution
6. **Persistence is built-in** - Easy to save and load indexes

---

## 🚀 Next Steps

**AP02 - Advanced Indexing Strategies**
- Learn about different index types
- Composable indexes
- Optimization strategies

**AP03 - Query Engines and Retrievers**
- Custom query engines
- Hybrid search
- Advanced retrieval

---

## 📖 Additional Resources

- [LlamaIndex Documentation](https://docs.llamaindex.ai/)
- [LlamaIndex Examples](https://github.com/run-llama/llama_index/tree/main/docs/examples)
- [LlamaIndex Discord](https://discord.gg/dGcwcsnxhU)
- [LangChain vs LlamaIndex Guide](../docs/langchain-vs-llamaindex.md)

---

*Created as part of the Applied AI module*  
*Master of Artificial Intelligence - University of Auckland*  
*Last updated: January 2025*